In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.ensemble import RandomForestClassifier

In [ ]:
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

# Cenário Completo
nome_treino = 'Sem Redução de Dimensionalidade/cicids2017_treinamento.csv'
nome_teste = 'Sem Redução de Dimensionalidade/cicids2017_teste.csv'

# Sem PortScan no treino
nome_treino_sem_portscan = 'Sem Redução de Dimensionalidade/cicids2017_treinamento_sem_portscan.csv'
nome_teste_com_portscan = 'Sem Redução de Dimensionalidade/cicids2017_teste_com_portscan.csv'

# Sem XSS no treino
nome_treino_sem_xss = 'Sem Redução de Dimensionalidade/cicids2017_treinamento_sem_XSS.csv'
nome_teste_com_xss = 'Sem Redução de Dimensionalidade/cicids2017_teste_com_XSS.csv'

# Outputs - Completo
nome_treino_reduzidos = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv'
nome_teste_reduzidos = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv'

# Outputs - Sem PortScan
nome_treino_reduzidos_sem_portscan = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_sem_portscan.csv'
nome_teste_reduzidos_com_portscan = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos_com_portscan.csv'

# Outputs - Sem XSS
nome_treino_reduzidos_sem_xss = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_sem_xss.csv'
nome_teste_reduzidos_com_xss = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos_com_xss.csv'

# Outputs - Balanced
nome_treino_reduzidos_balanced = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_balanceados.csv'
nome_teste_reduzidos_balanced = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos_balanceados.csv'

In [ ]:
print("Carregando os datasets tratados em CSV...")

df_treino = pd.read_csv(caminho_pasta_tratado + nome_treino)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_teste)

df_treino_sem_portscan = pd.read_csv(caminho_pasta_tratado + nome_treino_sem_portscan)
df_teste_com_portscan = pd.read_csv(caminho_pasta_tratado + nome_teste_com_portscan)

df_treino_sem_xss = pd.read_csv(caminho_pasta_tratado + nome_treino_sem_xss)
df_teste_com_xss = pd.read_csv(caminho_pasta_tratado + nome_teste_com_xss)

print(f"Completo     — Treino: {df_treino.shape} | Teste: {df_teste.shape}")
print(f"Sem PortScan — Treino: {df_treino_sem_portscan.shape} | Teste: {df_teste_com_portscan.shape}")
print(f"Sem XSS      — Treino: {df_treino_sem_xss.shape} | Teste: {df_teste_com_xss.shape}")

Carregando os datasets tratados em CSV...
Completo     — Treino: (1979513, 81) | Teste: (848363, 81)
Sem PortScan — Treino: (1868545, 81) | Teste: (959331, 81)
Sem XSS      — Treino: (1979054, 81) | Teste: (848822, 81)


In [ ]:
nomes_enviesados = {
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP',
    'Destination Port', 'Protocol', 'Timestamp', 'Fwd Header Length.1'
}

def remover_enviesadas(df_tr, df_te):
    colunas_para_remover = []
    fwd_header_length_vistos = 0
    for coluna in df_tr.columns:
        nome_normalizado = coluna.strip().lower()
        if coluna in nomes_enviesados:
            colunas_para_remover.append(coluna)
        if nome_normalizado == 'fwd header length':
            fwd_header_length_vistos += 1
            if fwd_header_length_vistos > 1:
                colunas_para_remover.append(coluna)
        if nome_normalizado.startswith('fwd header length.'):
            colunas_para_remover.append(coluna)
    colunas_para_remover = [c for c in dict.fromkeys(colunas_para_remover) if c != 'Label']
    df_tr = df_tr.drop(columns=[c for c in colunas_para_remover if c in df_tr.columns])
    df_te = df_te.drop(columns=[c for c in colunas_para_remover if c in df_te.columns])
    return df_tr, df_te

df_treino, df_teste = remover_enviesadas(df_treino, df_teste)
df_treino_sem_portscan, df_teste_com_portscan = remover_enviesadas(df_treino_sem_portscan, df_teste_com_portscan)
df_treino_sem_xss, df_teste_com_xss = remover_enviesadas(df_treino_sem_xss, df_teste_com_xss)

print(f"Completo     — Treino: {df_treino.shape} | Teste: {df_teste.shape}")
print(f"Sem PortScan — Treino: {df_treino_sem_portscan.shape} | Teste: {df_teste_com_portscan.shape}")
print(f"Sem XSS      — Treino: {df_treino_sem_xss.shape} | Teste: {df_teste_com_xss.shape}")

X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']
X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

X_treino_sem_portscan = df_treino_sem_portscan.drop('Label', axis=1)
y_treino_sem_portscan = df_treino_sem_portscan['Label']
X_teste_com_portscan = df_teste_com_portscan.drop('Label', axis=1)
y_teste_com_portscan = df_teste_com_portscan['Label']

X_treino_sem_xss = df_treino_sem_xss.drop('Label', axis=1)
y_treino_sem_xss = df_treino_sem_xss['Label']
X_teste_com_xss = df_teste_com_xss.drop('Label', axis=1)
y_teste_com_xss = df_teste_com_xss['Label']

Completo     — Treino: (1979513, 77) | Teste: (848363, 77)
Sem PortScan — Treino: (1868545, 77) | Teste: (959331, 77)
Sem XSS      — Treino: (1979054, 77) | Teste: (848822, 77)


In [ ]:
print("--- Cenário Completo: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_completo = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino.columns)} features | Selecionadas: {len(top_features_completo)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Completo: MDI ---
Treinamento RF concluído! Tempo: 63.4691s

Total: 76 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
11,Bwd Packet Length Mean,0.055883
51,Average Packet Size,0.053185
39,Packet Length Mean,0.051992
12,Bwd Packet Length Std,0.048513
41,Packet Length Variance,0.044265
64,Init_Win_bytes_forward,0.043264
40,Packet Length Std,0.042768
9,Bwd Packet Length Max,0.040962
3,Total Length of Fwd Packets,0.037278
53,Avg Bwd Segment Size,0.035490


In [ ]:
df_treino_reduzido = X_treino[top_features_completo].copy()
df_treino_reduzido['Label'] = y_treino.values
df_teste_reduzido = X_teste[top_features_completo].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos}")

Novas dimensões — Treino: (1979513, 52) | Teste: (848363, 52)
Treino salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv
Teste salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv


In [ ]:
print("--- Cenário Sem PortScan: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino_sem_portscan, y_treino_sem_portscan)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino_sem_portscan.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_sem_portscan = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino_sem_portscan.columns)} features | Selecionadas: {len(top_features_sem_portscan)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Sem PortScan: MDI ---
Treinamento RF concluído! Tempo: 56.8997s

Total: 76 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
12,Bwd Packet Length Std,0.070619
51,Average Packet Size,0.059418
41,Packet Length Variance,0.059344
64,Init_Win_bytes_forward,0.057303
11,Bwd Packet Length Mean,0.050327
40,Packet Length Std,0.050130
53,Avg Bwd Segment Size,0.047819
9,Bwd Packet Length Max,0.040735
61,Subflow Fwd Bytes,0.031438
7,Fwd Packet Length Mean,0.030720


In [ ]:
df_treino_reduzido = X_treino_sem_portscan[top_features_sem_portscan].copy()
df_treino_reduzido['Label'] = y_treino_sem_portscan.values
df_teste_reduzido = X_teste_com_portscan[top_features_sem_portscan].copy()
df_teste_reduzido['Label'] = y_teste_com_portscan.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_sem_portscan, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_sem_portscan}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_com_portscan, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_com_portscan}")

Novas dimensões — Treino: (1868545, 52) | Teste: (959331, 52)
Treino salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_sem_portscan.csv
Teste salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos_com_portscan.csv


In [ ]:
print("--- Cenário Sem XSS: MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
inicio_fs = time.time()
rf_fs.fit(X_treino_sem_xss, y_treino_sem_xss)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino_sem_xss.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_sem_xss = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino_sem_xss.columns)} features | Selecionadas: {len(top_features_sem_xss)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Sem XSS: MDI ---
Treinamento RF concluído! Tempo: 61.1498s

Total: 76 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
51,Average Packet Size,0.056650
11,Bwd Packet Length Mean,0.054522
40,Packet Length Std,0.051375
64,Init_Win_bytes_forward,0.047865
39,Packet Length Mean,0.047308
9,Bwd Packet Length Max,0.045843
12,Bwd Packet Length Std,0.043810
3,Total Length of Fwd Packets,0.041508
41,Packet Length Variance,0.038104
53,Avg Bwd Segment Size,0.036631


In [ ]:
df_treino_reduzido = X_treino_sem_xss[top_features_sem_xss].copy()
df_treino_reduzido['Label'] = y_treino_sem_xss.values
df_teste_reduzido = X_teste_com_xss[top_features_sem_xss].copy()
df_teste_reduzido['Label'] = y_teste_com_xss.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_sem_xss, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_sem_xss}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_com_xss, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_com_xss}")

Novas dimensões — Treino: (1979054, 52) | Teste: (848822, 52)
Treino salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_sem_xss.csv
Teste salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos_com_xss.csv


In [ ]:
print("--- Cenário Balanced (class_weight='balanced'): MDI ---")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')
inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()
print(f"Treinamento RF concluído! Tempo: {(fim_fs - inicio_fs):.4f}s")

df_importancias = pd.DataFrame({'Feature': X_treino.columns, 'Importancia': rf_fs.feature_importances_})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)
top_features_balanced = df_importancias.head(51)['Feature'].tolist()

print(f"\nTotal: {len(X_treino.columns)} features | Selecionadas: {len(top_features_balanced)}")
print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))

--- Cenário Balanced (class_weight='balanced'): MDI ---
Treinamento RF concluído! Tempo: 55.4468s

Total: 76 features | Selecionadas: 51

Top 10 features mais importantes:


,Feature,Importancia
65,Init_Win_bytes_backward,0.084826
64,Init_Win_bytes_forward,0.041466
53,Avg Bwd Segment Size,0.033042
38,Max Packet Length,0.032917
5,Fwd Packet Length Max,0.031299
51,Average Packet Size,0.030972
15,Flow IAT Mean,0.030597
16,Flow IAT Std,0.030563
19,Fwd IAT Total,0.028077
11,Bwd Packet Length Mean,0.027105


In [ ]:
df_treino_reduzido = X_treino[top_features_balanced].copy()
df_treino_reduzido['Label'] = y_treino.values
df_teste_reduzido = X_teste[top_features_balanced].copy()
df_teste_reduzido['Label'] = y_teste.values

print(f"Novas dimensões — Treino: {df_treino_reduzido.shape} | Teste: {df_teste_reduzido.shape}")
df_treino_reduzido.to_csv(caminho_pasta_tratado + nome_treino_reduzidos_balanced, index=False)
print(f"Treino salvo: {caminho_pasta_tratado + nome_treino_reduzidos_balanced}")
df_teste_reduzido.to_csv(caminho_pasta_tratado + nome_teste_reduzidos_balanced, index=False)
print(f"Teste salvo: {caminho_pasta_tratado + nome_teste_reduzidos_balanced}")

Novas dimensões — Treino: (1979513, 52) | Teste: (848363, 52)
Treino salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_balanceados.csv
Teste salvo: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos_balanceados.csv
